In [11]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(".")), "Code"))
from pathlib import Path


# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'LIS' / 'code'
mail_root = Path('/Users/jedrek/Library/Mail/V10')

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))


## Scan Mail for LIS result emails

Walks the entire macOS Mail store (`~/Library/Mail/V10`) and collects all `.emlx` messages from `postbox@lisdatacenter.org`, then displays them sorted chronologically.

In [12]:
import email
import email.policy
import re
import html
from html.parser import HTMLParser
from email.utils import parsedate_to_datetime

LIS_SENDER = "postbox@lisdatacenter.org"

def parse_emlx(path: Path):
    """Parse a macOS .emlx file (first line = byte count, then RFC 2822 email)."""
    try:
        raw = path.read_bytes()
        newline_pos = raw.index(b"\n")
        byte_count = int(raw[:newline_pos].strip())
        email_bytes = raw[newline_pos + 1 : newline_pos + 1 + byte_count]
        return email.message_from_bytes(email_bytes, policy=email.policy.default)
    except Exception:
        return None


def _decode_payload(part) -> str:
    """Decode a message part to a string, handling base64/QP encodings."""
    payload = part.get_payload(decode=True)
    if not payload:
        return ""
    charset = part.get_content_charset() or "utf-8"
    return payload.decode(charset, errors="replace")


class _HTMLToText(HTMLParser):
    """Convert HTML to readable plain text, similar to what a mail app shows."""

    BLOCK_TAGS = {"p", "div", "br", "li", "tr", "h1", "h2", "h3", "h4", "h5", "h6",
                  "blockquote", "pre", "hr", "table", "thead", "tbody", "tfoot"}
    SKIP_TAGS  = {"script", "style", "head"}

    def __init__(self):
        super().__init__(convert_charrefs=True)  # auto-converts &nbsp; &#160; etc.
        self._parts = []
        self._skip  = 0

    def handle_starttag(self, tag, attrs):
        tag = tag.lower()
        if tag in self.SKIP_TAGS:
            self._skip += 1
        elif tag in self.BLOCK_TAGS:
            self._parts.append("\n")

    def handle_endtag(self, tag):
        tag = tag.lower()
        if tag in self.SKIP_TAGS:
            self._skip = max(0, self._skip - 1)
        elif tag in self.BLOCK_TAGS:
            self._parts.append("\n")

    def handle_data(self, data):
        if self._skip == 0:
            self._parts.append(data)

    def get_text(self) -> str:
        text = "".join(self._parts)
        # Collapse runs of blank lines to at most two, strip trailing spaces per line
        lines = [l.rstrip() for l in text.splitlines()]
        cleaned = []
        blank_run = 0
        for line in lines:
            if line == "":
                blank_run += 1
                if blank_run <= 1:
                    cleaned.append("")
            else:
                blank_run = 0
                cleaned.append(line)
        return "\n".join(cleaned).strip()


def _html_to_text(html_str: str) -> str:
    parser = _HTMLToText()
    parser.feed(html_str)
    return parser.get_text()


def get_body(msg) -> str:
    """Extract the best plain-text body. Prefers text/plain; falls back to text/html."""
    plain_parts = []
    html_parts  = []

    if msg.is_multipart():
        for part in msg.walk():
            if part.is_multipart():
                continue
            if "attachment" in str(part.get("Content-Disposition", "")):
                continue
            ct = part.get_content_type()
            if ct == "text/plain":
                plain_parts.append(_decode_payload(part))
            elif ct == "text/html":
                html_parts.append(_decode_payload(part))
    else:
        ct = msg.get_content_type()
        if ct == "text/plain":
            plain_parts.append(_decode_payload(msg))
        elif ct == "text/html":
            html_parts.append(_decode_payload(msg))

    if plain_parts:
        return "\n".join(plain_parts).strip()
    if html_parts:
        return _html_to_text("\n".join(html_parts))

    raw = msg.get_payload(decode=True)
    if raw:
        return raw.decode("utf-8", errors="replace").strip()
    return ""


records = []

for emlx_path in mail_root.rglob("*.emlx"):
    if emlx_path.name.endswith(".partial.emlx"):
        continue
    msg = parse_emlx(emlx_path)
    if msg is None:
        continue

    sender = str(msg.get("From", ""))
    if LIS_SENDER not in sender.lower():
        continue

    date_str = str(msg.get("Date", ""))
    try:
        dt = parsedate_to_datetime(date_str)
    except Exception:
        dt = None

    records.append({
        "date":    dt,
        "subject": str(msg.get("Subject", "(no subject)")),
        "body":    get_body(msg),
        "file":    str(emlx_path),
    })

print(f"Found {len(records)} email(s) from {LIS_SENDER}")


Found 59 email(s) from postbox@lisdatacenter.org


In [13]:
import pandas as pd

df = pd.DataFrame(records)

if df.empty:
    print("No emails found.")
else:
    # Sort chronologically (NaT dates go last)
    df = df.sort_values("date", na_position="last").reset_index(drop=True)
    df.index += 1  # 1-based numbering

    # Display summary table
    display(df[["date", "subject"]].rename(columns={"date": "Date", "subject": "Subject"}))


,Date,Subject
1,2026-01-31 13:15:57+01:00,job 1442359 PL_1
2,2026-01-31 13:20:12+01:00,job 1442364 PL_2
3,2026-01-31 13:21:34+01:00,job 1442365 PL_3
4,2026-01-31 13:36:01+01:00,job 1442373 PL_4
5,2026-01-31 13:39:36+01:00,job 1442374 PL_4
6,2026-01-31 15:58:34+01:00,job 1442400 PL_5
7,2026-01-31 16:01:40+01:00,job 1442402 PL_5
8,2026-01-31 16:02:43+01:00,job 1442403 PL_51
9,2026-01-31 16:06:48+01:00,job 1442404 PL_51
10,2026-01-31 16:37:49+01:00,job 1442408 PL_levels


### Inspect a specific email body

Change `EMAIL_INDEX` (1-based) to read the full body of any listed email.

In [ ]:
df.sort_values("date", ascending=False, inplace=True)

In [19]:
EMAIL_INDEX = 1  # change to inspect a different email (1-based)

if not df.empty and EMAIL_INDEX <= len(df):
    row = df.iloc[EMAIL_INDEX - 1]
    print(f"Date   : {row['date']}")
    print(f"Subject: {row['subject']}")
    print(f"File   : {row['file']}")
    print("-" * 60)
    print(row["body"])
else:
    print("No email at that index.")

Date   : 2026-03-07 15:54:03+01:00
Subject: job 1453264 Test job from LISAutomizer
File   : /Users/jedrek/Library/Mail/V10/67AF7211-575D-461C-848B-04F9ED568CC1/INBOX.mbox/4E96815A-30EB-42EF-B40B-C98AD2E7740C/Data/9/2/Messages/29759.emlx
------------------------------------------------------------
############################### NOTICE TO USERS ###############################
                                                                        Use of the data in the LUXEMBOURG INCOME STUDY DATABASES is governed by regulations which do not allow copying or further distribution of the survey microdata.

Anyone violating these regulations will lose all privileges to the databases and may be subject to prosecution under the law. In addition, any attempt to circumvent the LIS processing system or unauthorized entry into the LIS computing system will result in prosecution.
All papers written using the LUXEMBOURG INCOME STUDY DATABASES must be  submitted for entry into the Working Papers Se